<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/Pinecone_vectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Vectors, the output data format of Neural Network models, can effectively encode information and serve a pivotal role in AI applications such as knowledge base, semantic search, Retrieval Augmented Generation (RAG) and more.

Pinecone is a managed, cloud-native vector database designed for high performance and scalability. In this guide, we will walk you through how to use Pinecone to store and search vectors in the cloud.

## Install Dependencies
First, we will install the required libraries: `pinecone-client`, `sentence-transformers`, `pandas`, and `python-dotenv`.

In [4]:
!pip install pinecone sentence-transformers pandas python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0


## Initialize Pinecone Client
To use Pinecone, you need an API key from the Pinecone console. Create yet another account in Pinecone and generate an API key that can be added to the google colab secrets.

In [4]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata





In [9]:
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_KEY')

### Creating and using the HUGGINGFACE_TOKEN is optional (as of now in colab), but I would recommend that you do it, as it is used whenever SENTENCE TRANSFORMERS  models will be called.
os.environ["HF_TOKEN"] = userdata.get('HUGGINGFACE_TOKEN')

# Initialize the Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

## Set Up Embedding Model
We will use `sentence-transformers` to embed the text data. This model generates 384-dimensional embeddings.

In [7]:
from sentence_transformers import SentenceTransformer

# Initialize the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Create a Pinecone Index
We need an index in Pinecone to store our vectors. The dimension parameter must match the size of embeddings generated by our model (384-dimensions).

In [8]:
index_name = "is-research"

# Check if the index exists, and if not, create it
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, # Match the 'all-MiniLM-L6-v2' output dimension
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

# Connect to the index
index = pc.Index(index_name)

## Prepare and Load Data
Load our existing dataset, `ISResearch.csv`, into a Pandas DataFrame. Since Pinecone operates on the cloud, we will embed the abstracts locally and push batches of `(id, vector, metadata)` directly to the index.

In [11]:
import pandas as pd

# Load your CSV
df = pd.read_csv("https://raw.githubusercontent.com/KarAnalytics/datasets/refs/heads/master/ISResearch.csv")

df.head()

,id,Year,Title,Abstract,URL,JournalFN
0,1,2024,Digital Approaches to Societal Grand Challenge...,Information systems (IS) scholars have pursued...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
1,2,2024,Mr. Right or Mr. Best: The Role of Information...,This paper examines the role of information in...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
2,3,2024,How Information Technology Overcomes Deficienc...,Innovation is vital for the growth of small an...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
3,4,2024,Strategic Expectation Setting of Delivery Time...,Delivery speed is an essential component of th...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
4,5,2024,User-Generated Content Shapes Judicial Reasoni...,Legal professionals have access to many differ...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research


In [12]:
# Generate embeddings for all abstracts
abstracts = df['Abstract'].tolist()
embeddings = model.encode(abstracts)

# Prepare data in the format Pinecone requires: a list of tuples (id, vector, metadata_dict)
vectors_to_upsert = []
for i, row in df.iterrows():
    vectors_to_upsert.append({
        "id": str(row['id']),
        "values": embeddings[i].tolist(),
        "metadata": {
            "Title": row['Title'],
            "Year": row['Year'],
            "URL": row['URL'],
            "Journal": row['JournalFN']
        }
    })

# Upsert (insert or update) the vectors into Pinecone in batches to avoid payload limits
batch_size = 100
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i:i + batch_size]
    index.upsert(vectors=batch)

print(f"Migration Complete. Upserted {len(vectors_to_upsert)} vectors to Pinecone.")

Migration Complete. Upserted 227 vectors to Pinecone.


## Semantic Search
Now we can perform semantic searches. First, we generate an embedding for our search query and then we send it to Pinecone to retrieve the top matches along with their metadata.

In [13]:
query = "Which papers mention blockchain or decentralized systems?"

# Embed the query text
query_vector = model.encode([query])[0].tolist()

# Search the Pinecone index
results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True # Ensure we retrieve the Title, Year, and URL
)

print("\n--- Top 5 Semantic Search Results ---")
for match in results['matches']:
    metadata = match['metadata']
    print(f"Score: {match['score']:.4f} | Year: {metadata.get('Year')} | Title: {metadata.get('Title')}")

# Optional: To show the URL of the top result
top_match = results['matches'][0]['metadata']
print(f"\nTop Match URL: {top_match.get('URL')}")


--- Top 5 Semantic Search Results ---
Score: 0.4419 | Year: 2024 | Title: Skin in the Game: The Transformational Potential of Decentralized Autonomous Organizations
Score: 0.3866 | Year: 2025 | Title: Foundations of Decentralized Metaverse Economies: Converging Physical and Virtual Realities
Score: 0.3589 | Year: 2024 | Title: Digitization of Transaction Terms within TCE: Strong Smart Contract as a New Mode of Transaction Governance
Score: 0.3269 | Year: 2024 | Title: Digital Transformation of Academic Publishing:  A Call for the Decentralization and Democratization of Academic Journals
Score: 0.3133 | Year: 2025 | Title: Uncovering the Structural Assurance Mechanisms  in Blockchain Technology-Enabled Online Healthcare Mutual Aid Platforms

Top Match URL: https://misq.umn.edu/skin-the-the-game-the-transformational-potential-of-decentralized-autonomous-organizations.html
